# Local SQL queries

The app keeps its users in a small SQLite file: `data/app.db`. `seed.py` creates it, `main.py` reads it.

This notebook opens the same file and shows what the queries look like. Run it from the `05_user_rag` folder.

Nothing extra to install: `sqlite3` ships with Python.

In [1]:
import sqlite3
from pathlib import Path

# Same file that seed.py creates and main.py reads
DB_PATH = Path("data/app.db")

if not DB_PATH.exists():
    raise FileNotFoundError("data/app.db is missing. Run `python seed.py` in this folder first.")

print(DB_PATH.resolve())

/home/anuarsantoyo/Projects/Projects/MasterSchool/MLApp_ProdEng_Oct26/ProdEng/05_user_rag/data/app.db


## 1. What is inside the file

In [2]:
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row  # lets us read columns by name

schema = conn.execute(
    "SELECT sql FROM sqlite_master WHERE type = 'table' AND name = 'users'"
).fetchone()[0]

print(schema)

CREATE TABLE users (
    user_id INTEGER PRIMARY KEY,
    user_name TEXT NOT NULL,
    country TEXT NOT NULL,
    role TEXT NOT NULL
)


## 2. Read every row

In [8]:
rows = conn.execute("SELECT user_id, user_name, country, role FROM users").fetchall()

for row in rows:
    print(dict(row))

{'user_id': 1, 'user_name': 'Ana Torres', 'country': 'Mexico', 'role': 'owner'}
{'user_id': 2, 'user_name': 'Luis Mendez', 'country': 'Spain', 'role': 'billing_manager'}
{'user_id': 3, 'user_name': 'Priya Nair', 'country': 'India', 'role': 'agent'}
{'user_id': 4, 'user_name': 'Emma Chen', 'country': 'Singapore', 'role': 'admin'}
{'user_id': 5, 'user_name': 'Diego Ruiz', 'country': 'Germany', 'role': 'contractor'}


In [9]:
rows

## 3. Read one row, the way the API does it

`main.py` takes the `user_id` from the request body and asks for that one row. The `?` is a placeholder: the value is passed separately, so it can only ever be a value and never part of the query text.

In [15]:
user_id = 3

row = conn.execute(
    "SELECT user_id, user_name, country, role FROM users WHERE user_id = ?",
    (user_id,),
).fetchone()

#print(dict(row) if row else None)

In [16]:
row

## 4. Filter on a column

In [17]:
rows = conn.execute(
    "SELECT user_name, role FROM users WHERE country = ? ORDER BY user_name",
    ("Mexico",),
).fetchall()

for row in rows:
    print(dict(row))

{'user_name': 'Ana Torres', 'role': 'owner'}


## 5. When there is no row

In [18]:
row = conn.execute("SELECT * FROM users WHERE user_id = ?", (999,)).fetchone()

print(row)  # None, and main.py turns that into a 404

None


In [19]:
conn.close()

That is the whole database layer. `seed.py` opens the same file to run `CREATE TABLE IF NOT EXISTS` and `INSERT OR IGNORE`, and `main.py` wraps the query from section 3 inside `get_user()`.